# 全國教師在職進修資訊網 - 線上課程蒐集 (Google Colab 版)
本腳本將本地端的所有爬蟲、過濾、與 LLM 驗證程式碼全部遷移至 Colab 執行。
請依序點擊每一個儲存格左側的「播放(Play)」按鈕即可。

### 步驟 1: 環境安裝 (約需 1 分鐘)

In [ ]:
!apt-get update -y
!apt-get install -y xvfb
# 安裝 Chromium 所需的系統依賴（Colab 有時缺少這些）
!apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
    libxkbcommon0 libatspi2.0-0 libxcomposite1 libxdamage1 libxfixes3 \
    libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2 \
    libnss3 libnspr4 2>/dev/null || true
!pip install -q playwright nest-asyncio
!playwright install chromium
!playwright install-deps chromium

print("環境安裝完成！")


### 步驟 2: 啟動虛擬螢幕模組與設定參數
這裡可以設定您想要爬取的參數 (開始日期、天數、API Key 等)。

In [ ]:
import os
from datetime import date, datetime, timedelta

# 啟動 Xvfb 並設定虛擬螢幕變數 (非常重要，用來騙過防爬蟲機制)
os.system('Xvfb :99 -screen 0 1024x768x24 &')
os.environ['DISPLAY'] = ':99'

# --- 參數設定 ---
SEARCH_START_DATE = "2026-04-26" # @param {type:"date"}
SEARCH_DAYS = 3 # @param {type:"integer"}
IT_KEYWORDS = ['資訊科技', 'AI', '人工智慧', 'Canva', '程式設計', '數位', '資安', '軟體', '電腦', '網路', '機器人']

# 解析開始日期（若為空或無效則用今天）
try:
    _start = datetime.strptime(SEARCH_START_DATE, '%Y-%m-%d').date()
except:
    _start = date.today()

SEARCH_START = _start
SEARCH_END = _start + timedelta(days=SEARCH_DAYS)

print(f"虛擬螢幕已啟動，參數設定完畢。")
print(f"搜尋範圍：{SEARCH_START.strftime('%Y/%m/%d')} ~ {SEARCH_END.strftime('%Y/%m/%d')}")

### 步驟 3: 執行 Playwright 爬蟲 (透過進階搜尋抓取指定日期範圍的課程)

In [ ]:
import asyncio
import nest_asyncio
import csv
import re
from playwright.async_api import async_playwright

nest_asyncio.apply()

# ── 進階搜尋常數 ─────────────────────────────────────────────────
ADVANCED_URL = "https://www2.inservice.edu.tw/NAPP/OpenQuery.aspx"
SEL_DATE_START = "#ctl00_CPH_Content_RDP_Start_dateInput"
SEL_DATE_END   = "#ctl00_CPH_Content_RDP_End_dateInput"
SEL_SEARCH_BTN = "button[type='submit']"
SEL_PAGER_INFO = ".rgInfoPart"
SEL_NEXT_BTN   = "button.rgPageNext"


async def fill_date(page, selector, date_str):
    """填入 Telerik RadDatePicker 日期欄位"""
    loc = page.locator(selector).first
    await loc.click()
    await page.wait_for_timeout(300)
    await page.keyboard.press('Control+A')
    await page.keyboard.press('Delete')
    await loc.type(date_str, delay=60)
    await page.keyboard.press('Tab')
    await page.wait_for_timeout(400)


async def set_page_size_50(page):
    """將每頁筆數改為 50（Telerik RadComboBox）"""
    try:
        combo_arrow = page.locator('.rgAdvPart .rcbActionButton').first
        if await combo_arrow.count() == 0:
            return False
        await combo_arrow.click()
        await page.wait_for_timeout(800)

        item_50 = page.locator('.rgAdvPart .rcbList .rcbItem').filter(has_text='50').first
        if await item_50.count() > 0:
            await item_50.click()
        else:
            await page.locator("li.rcbItem:has-text('50')").first.click()

        await page.wait_for_timeout(2000)
        try:
            await page.wait_for_load_state('networkidle', timeout=10000)
        except:
            pass
        await page.wait_for_timeout(1000)
        return True
    except Exception as e:
        print(f'  (無法調整每頁筆數: {e})')
        return False


async def extract_table_rows(page):
    """擷取 rgMasterTable 表格，回傳 (header_or_None, data_rows)"""
    rows = await page.eval_on_selector_all(
        'table.rgMasterTable tr',
        """(trs) => trs.map(tr => {
            const cells = Array.from(tr.querySelectorAll('th, td'));
            return cells.map(c => c.innerText.trim().replace(/[\\n,]/g, ' '));
        })"""
    )
    if not rows:
        return None, []
    header = None
    data_start = 0
    for i, r in enumerate(rows):
        if any('課程代碼' in cell for cell in r):
            header = r
            data_start = i + 1
            break
    data_rows = [
        r for r in rows[data_start:]
        if len(r) >= 4 and r[0] and r[0][0].isdigit()
    ]
    return header, data_rows


# ── 主爬蟲 ───────────────────────────────────────────────────────
all_data = []      # 全部課程列表 (含 header)
filtered_final = [] # 本次不再做日期過濾（進階搜尋已限定範圍）

async def scrape_courses():
    global all_data, filtered_final
    start_str = SEARCH_START.strftime('%Y/%m/%d')
    end_str   = SEARCH_END.strftime('%Y/%m/%d')

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=['--disable-blink-features=AutomationControlled',
                  '--no-sandbox', '--disable-setuid-sandbox',
                  '--disable-dev-shm-usage']
        )
        context = await browser.new_context(
            viewport={'width': 1280, 'height': 900},
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
        )
        page = await context.new_page()

        # Step 1: 進入進階搜尋
        print(f'[1] 前往進階搜尋頁...')
        await page.goto(ADVANCED_URL, wait_until='networkidle', timeout=30000)
        await page.wait_for_timeout(2000)

        # Step 2: 填入日期
        print(f'[2] 填入日期範圍：{start_str} ~ {end_str}')
        await fill_date(page, SEL_DATE_START, start_str)
        await fill_date(page, SEL_DATE_END, end_str)

        # Step 3: 點查詢
        print('[3] 點擊「查詢」...')
        await page.locator(SEL_SEARCH_BTN).first.click()
        await page.wait_for_timeout(3000)

        # Step 4: 讀取總筆數
        total_pages = 999
        try:
            pager_text = await page.locator(SEL_PAGER_INFO).first.text_content()
            print(f'[4] 搜尋結果：{pager_text.strip()}')
            m = re.search(r'(\d+)\s+items?\s+in\s+(\d+)\s+pages?', pager_text)
            if m:
                total_pages = int(m.group(2))
        except:
            print('[4] 無法取得總筆數')

        # Step 5: 設定每頁 50 筆
        print('[5] 調整每頁筆數為 50...')
        await set_page_size_50(page)
        try:
            pager_text = await page.locator(SEL_PAGER_INFO).first.text_content()
            print(f'    調整後：{pager_text.strip()}')
            m = re.search(r'(\d+)\s+items?\s+in\s+(\d+)\s+pages?', pager_text)
            if m:
                total_pages = int(m.group(2))
        except:
            pass

        # Step 6: 逐頁爬取
        data_rows_all = []
        header = None
        page_num = 1
        prev_first_code = None

        while page_num <= total_pages:
            print(f'  第 {page_num}/{total_pages} 頁...', end='')
            try:
                await page.wait_for_selector('table.rgMasterTable', timeout=15000)
            except:
                print(' 找不到表格，結束'); break

            h, data_rows = await extract_table_rows(page)
            if header is None and h:
                header = h
            if not data_rows:
                print(' 無資料，結束'); break

            curr_first_code = data_rows[0][0] if data_rows else ''
            if prev_first_code and curr_first_code == prev_first_code:
                print(' 偵測到重複頁面，結束'); break
            prev_first_code = curr_first_code

            data_rows_all.extend(data_rows)
            print(f' {len(data_rows)} 筆（累計 {len(data_rows_all)}）')

            if page_num >= total_pages:
                break

            try:
                next_btn = page.locator(SEL_NEXT_BTN).first
                if await next_btn.count() == 0 or not await next_btn.is_enabled():
                    break
                await next_btn.click()
                await page.wait_for_timeout(1500)
                try:
                    await page.wait_for_function(
                        f"""() => {{
                            const td = document.querySelector('table.rgMasterTable tr:nth-child(2) td');
                            return td && td.innerText.trim() !== '{curr_first_code}';
                        }}""",
                        timeout=10000
                    )
                except:
                    await page.wait_for_timeout(2000)
                page_num += 1
            except Exception as e:
                print(f'  翻頁異常: {e}'); break

        await browser.close()

    # 組合結果（相容下游步驟）
    if header is None:
        header = ['課程代碼', '研習名稱', '開始日期', '結束日期', '辦理研習單位']
    all_data = [header] + data_rows_all
    # 進階搜尋已限定日期範圍，不需再做日期過濾
    filtered_final = list(all_data)

    print(f'\n爬蟲完畢！共 {len(data_rows_all)} 筆課程。')

await scrape_courses()


### 步驟 4: 主題過濾 (挑出 IT/AI 相關課程)

In [ ]:
try:
    from google.colab import ai
except ImportError:
    print("無法匯入 google.colab.ai。提醒：這只能在 Google Colab 環境下執行。")

import json
import re

it_courses = [filtered_final[0]] # header
courses_to_check = [row for row in filtered_final[1:] if len(row) > 2]
print(f"準備對 {len(courses_to_check)} 筆課程進行 LLM 主題過濾...")

# 每次處理 50 筆，避免 LLM 偷懶或超過字數限制
chunk_size = 50
for i in range(0, len(courses_to_check), chunk_size):
    chunk = courses_to_check[i:i+chunk_size]

    course_list_text = ""
    for j, row in enumerate(chunk):
        # print(row[1])
        course_list_text += f"{j}. {row[1]}\n"

    prompt = f"""在列表中逐一掃描研習名稱，找出與以下主題相關的課程：
- 資訊科技、程式設計
- AI 相關 (例如：Gemini, NotebookLM 等，但不限於列出的這些)
- Canva 設計
- 機電整合、生活科技
- STEAM

請嚴格以純 JSON 格式回傳符合條件的「課程編號」陣列，例如：[0, 2, 4]。不要加上 ```json 標籤或是任何其他解釋！
如果沒有符合的課程，請回傳 []。

課程列表如下：
{course_list_text}
"""
    try:
        print(f"傳送第 {i+1} ~ {min(i+chunk_size, len(courses_to_check))} 筆至 LLM 分析...")
        response = ai.generate_text(prompt)
        response_text = str(response).replace('`json', '').replace('`', '').strip()

        # 嘗試用 Regex 抓取陣列結構
        json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
        if json_match:
            indices = json.loads(json_match.group(0))
        else:
            indices = json.loads(response_text)

        if not isinstance(indices, list):
            raise ValueError("LLM 沒有回傳有效的陣列格式。")

        for idx in indices:
            if isinstance(idx, int) and 0 <= idx < len(chunk):
                it_courses.append(chunk[idx])
                print(f"  ✅ 保留 [{i+idx}]: {chunk[idx][1]}")

    except Exception as e:
        print(f"  ! LLM 解析失敗 (該批次全部預設保留): {e}")
        try:
             print(f"  > LLM 回應結果：{response_text[:100]}...")
        except: pass
        it_courses.extend(chunk)

print(f"\n經過 LLM 過濾，符合上述主題的課程有 {len(it_courses)-1} 筆。")


### 步驟 5: 爬蟲二次驗證 (進入課程頁面尋找線上視訊連結)

In [ ]:
import json

valid_results = []

async def verify_courses():
    global valid_results
    if len(it_courses) > 1:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False, args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage'])
            context = await browser.new_context(user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36')
            page = await context.new_page()

            for i, row in enumerate(it_courses[1:]):
                if len(row) < 2: continue
                cid = row[0]   # 新版：課程代碼在 index 0
                name = row[1]  # 新版：研習名稱在 index 1

                print(f"正在驗證 [{i+1}/{len(it_courses)-1}] {name}...")
                url = f"https://www2.inservice.edu.tw/NAPP/CourseView.aspx?cid={cid}"

                try:
                    await page.goto(url, timeout=30000)
                    await asyncio.sleep(2)

                    page_text = await page.inner_text("body")
                    content = await page.content()

                    is_online = any(term.lower() in page_text.lower() for term in ['meet.google.com', 'teams.microsoft.com', 'zoom.us', 'webex', '數位遠距教學', '線上研習', '視訊連結', 'meet'])

                    has_maps_link = 'google.com/maps' in content or 'maps.google.com' in content
                    onsite_location = False
                    location_text = ""
                    try:
                        if "開課地點：" in page_text:
                            match = re.search(r'開課地點：(.*)', page_text)
                            if match:
                                location_text = match.group(1).strip()
                    except: pass

                    if location_text and any(p in location_text for p in ['路', '號', '里', '區', '電腦教室', '室', '校']):
                        if "線上" not in location_text and "數位" not in location_text:
                            onsite_location = True

                    is_onsite = any(term in page_text for term in ['實體教室', '現場參加', '僅限校內教師參加', '未開放線上報名', '現場報名', '報到地點', '本校教室'])
                    if has_maps_link: is_onsite = True
                    if is_online: is_onsite = False # Meeting link overrides

                    if is_online and not is_onsite:
                        valid_results.append({
                            "cid": cid,
                            "name": name,
                            "time": row[2] if len(row) > 2 else "Unknown",  # 新版：開始日期在 index 2
                            "raw_text": page_text[:2000]
                        })
                        print("  -> 驗證通過：是純線上課程！")
                    else:
                        print("  -> 過濾：實體課程或非線上課程。")

                except Exception as e:
                     print(f"  -> Error checking {cid}: {e}")

            await browser.close()

    print(f"\n第一階段驗證完畢，共有 {len(valid_results)} 堂純線上課程準備交給 Gemini 分析。")

await verify_courses()


### 步驟 6: 呼叫 Gemini AI 精準擷取時間/主講人，生成 Markdown 報告

In [ ]:
# 取代原本需 API key 的寫法，改由 Colab 內建的 ai 模組直接呼叫
try:
    from google.colab import ai
except ImportError:
    print("無法匯入 google.colab.ai。提醒：這只能在 Google Colab 環境下執行。")

import json
from datetime import datetime

# 使用搜尋起始日期作為報表檔名
# date_stamp = SEARCH_START.strftime('%Y%m%d')
# output_file = f"courselist_{date_stamp}_final_report.md"
search_end_date_stamp = SEARCH_END.strftime('%Y%m%d')
output_file = f"courselist_{search_end_date_stamp}_final_report.md"

if len(valid_results) == 0:
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 全國教師在職進修資訊網 - 線上課程清單 ({SEARCH_START.strftime('%Y/%m/%d')} ~ {SEARCH_END.strftime('%Y/%m/%d')})\n\n")
        f.write("本次範圍內無符合條件之課程。\n")
    print("無符合的課程，已生成空白報表。")
else:
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 全國教師在職進修資訊網 - 線上課程清單 ({SEARCH_START.strftime('%Y/%m/%d')} ~ {SEARCH_END.strftime('%Y/%m/%d')})\n\n")
        f.write("> [!NOTE]\n")
        f.write("> **AI 語意分析版本**：本報告由 Colab 內建 google.colab.ai (無須 API key) 擷取資料。\n\n")
        f.write("| 課程代碼 | 課程名稱 | 研習時間 (起訖) | Google Meet 網址 / 線上連結 | 主講人 |\n")
        f.write("| :--- | :--- | :--- | :--- | :--- |\n")

        for i, c in enumerate(valid_results):
            print(f"請 Gemini 分析 [{i+1}/{len(valid_results)}] {c['name']} ...")

            prompt = f"""請作為課程擷取員，從以下文字萃取資訊並返回 JSON 物件。
【提取規則】：
1. time: 完整上課具體起訖時段 (如 2026/04/07(二) 13:20~16:30)，民國年請轉西元年。
2. speaker: 主講人或講師姓名 (拔除身分標籤)，無則填未提供。
3. url: Meet或視訊網址，無則填請見內文。
請嚴格使用不帶任何格式或markdown語法的純 JSON 回傳，格式: {{\"time\": \"...\", \"speaker\": \"...\", \"url\": \"...\"}}

課程原始文字：
{c['raw_text']}
"""
            try:
                # The correct syntax provided by user!
                response = ai.generate_text(prompt)
                response_text = str(response)

                # 清理與擷取 JSON
                response_text = response_text.replace('`json', '').replace('`', '').strip()
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    data = json.loads(json_match.group(0))
                else:
                    data = json.loads(response_text)

                time_str = data.get("time", c.get('time', '未提供'))
                speaker_str = data.get("speaker", "未提供")
                url_str = data.get("url", "請見內文")
            except Exception as e:
                print(f"  ! 擷取失敗: {e}")
                time_str = c.get('time', '未提供')
                speaker_str = "備註：自動擷取失敗"
                url_str = "請見內文"

            official_url = f"https://www2.inservice.edu.tw/NAPP/CourseView.aspx?cid={c['cid']}"
            f.write(f"| [{c['cid']}]({official_url}) | {c['name']} | **{time_str}** | {url_str} | {speaker_str} |\n")

    print(f"\n✅ 報表生成完畢：{output_file}")


### 步驟 7: 更新資料回 GitHub
這一步驟可以自動將產出的報表推送到你的 GitHub 專案中

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

# 1. 設定 Git 使用者資訊
!git config --global user.email "jefffang.edu@gmail.com"
!git config --global user.name "JeffCodingMentor"

# 2. Clone 專案 (建議使用 GitHub Personal Access Token)
# 格式為 https://<TOKEN>@github.com/JeffCodingMentor/inspage.git
!git clone https://{token}@github.com/JeffCodingMentor/inspage.git

# 3. 將產生的新 md 檔移動/複製到 inspage/data 目錄下
!cp courselist_*.md inspage/data/

# 4. 提交並推送到 GitHub
%cd inspage
!git add data/*.md
!git commit -m "Auto-update course data from Colab"
!git push origin main
